# One Look — Stage 3: run every experiment, then the demo

Run this after `01_setup_data.ipynb` (data on Google Drive). Run the cells **in order**.
It takes about 5–10 minutes. Everything is saved to `MyDrive/one-look/outputs/`.

What it does:
1. Labels every chip and makes the geographic split (test = Bolivia + Ghana).
2. Compares 4 approaches: MNDWI rule, logistic regression, small and large random forest.
3. Builds the UNKNOWN detector and tests it.
4. Measures model size and speed, and lists every wrong decision (failure analysis).
5. Demo on a test chip the model never saw, and on your own Sentinel-2 image.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA = '/content/drive/MyDrive/one-look/data/HandLabeled'
OUT  = '/content/drive/MyDrive/one-look'
%cd /content
!rm -rf One-Look && git clone -q https://github.com/ojaswi-raya/One-Look.git
%cd /content/One-Look
!pip -q install -r requirements.txt

## 1. Labels and geographic split

In [ ]:
!python src/labels.py --root "$DATA" --out "$OUT/data/labels.csv"
!python src/split.py  --labels "$OUT/data/labels.csv" --out "$OUT/data/splits.json"

## 2–4. All experiments (approach comparison, UNKNOWN detector, size/speed, failures)

In [ ]:
!python src/run_experiments.py --root "$DATA" --labels "$OUT/data/labels.csv" --splits "$OUT/data/splits.json" --out "$OUT/outputs"

In [ ]:
from IPython.display import Markdown, Image, display
import pandas as pd
display(Markdown(open(f'{OUT}/outputs/results.md').read()))

### Failure analysis: every wrong decision on the test set

In [ ]:
display(pd.read_csv(f'{OUT}/outputs/failures.csv'))
display(Image(f'{OUT}/outputs/failures.png'))

## 5. Demo on an image the model has never seen
First a Bolivia chip (test country, never used for training or tuning).

In [ ]:
import glob, os
chip = sorted(glob.glob(f'{DATA}/S2Hand/Bolivia_*_S2Hand.tif'))[0]
!python src/predict.py --image "$chip" --model "$OUT/outputs/one_look_model.joblib" --out "$OUT/outputs/demo_test_chip.png"
display(Image(f'{OUT}/outputs/demo_test_chip.png'))

### Demo on your own new image
Upload a Sentinel-2 **L1C** GeoTIFF with all **13 bands** (B1, B2, B3, B4, B5, B6, B7, B8, B8A, B9, B10, B11, B12)
at 10 m, downloaded from Copernicus Browser. Large scenes are cut into 512×512 tiles, one decision per tile.

In [ ]:
from google.colab import files
up = files.upload()
for name in up:
    !python src/predict.py --image "$name" --model "$OUT/outputs/one_look_model.joblib" --out "$OUT/outputs/demo_{name}.png"
    display(Image(f'{OUT}/outputs/demo_{name}.png'))

## 6. Put the results in your GitHub repo
This downloads a zip of `outputs/`. Unzip it and upload the files into a `results/` folder of the repo
(GitHub → Add file → Upload files), then paste `results.md` into the README results section.

In [ ]:
!cd "$OUT" && zip -qr /content/results.zip outputs
files.download('/content/results.zip')